# Enriching Archival Metadata for People Discovery

<a href="https://colab.research.google.com/github/programminghistorian/ph-submissions/blob/gh-pages/assets/enablar-lesson-6/enablar-lesson-6-workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This Jupyter Notebook will allow you to follow the lesson on the drive rather than locally.
For the purpose of the workshop, we will demonstrate how to extract text data from PDFs for Named Entity Recognition.

### Installing required libraries and models

You will need to make sure that the following libraries are installed in the Google Colab. Some of these might be installed already, but it can't hurt to install them again.

In [ ]:
#installing libraries and models needed for workshop
%pip install pdfplumber
%pip install pandas
%pip install spacy
!python -m spacy download en_core_web_lg

### 1. Extracting Text with pdfplumber

In [ ]:
#importing libraries needed for this section
import pdfplumber
import pandas as pd
from pathlib import Path
import zipfile

In [ ]:
#using zipfile to access our data
with zipfile.ZipFile("/content/enablar_lesson_6.zip", "r") as zip_file:
  zip_file.extractall("AA_Weekly_data")

In [ ]:
#function to get all the pdf files in a givendirectory
def get_pdfs(dir): #dir refers to directory
   files = []
   #the asterisk * below is a wildcard, meaning what comes before the file extension does not matter
   for path in Path(dir).glob("*.pdf"):
       files.append(path)
   return files

In [ ]:
#get all files from our unzipped directory
aa_files = get_pdfs("AA_Weekly_data")

In [ ]:
#function to extract text from pdf files
def pdf_to_df(files):
   rows = []
   for doc in files:
       with pdfplumber.open(doc) as pdf:
           full_text = []
           for page in pdf.pages:
               text = page.extract_text()
               if text:
                   full_text.append(text)
           combined_pdf = " ".join(full_text)
       #append to rows and use Path to only keep the file name
       rows.append({"file_name": Path(doc).name, "text": combined_pdf})
       #turn into a DataFrame
   df = pd.DataFrame(rows)
   return df


In [ ]:
#apply the function and check it worked by printing
aa_files_data = pdf_to_df(aa_files)
print(aa_files_data['text'])

### 2. Running the NER

In [ ]:
import spacy
#load spacy moodel
nlp = spacy.load("en_core_web_lg")


In [ ]:
#ner function
def run_ner(text, nlp): #arguments are text and the nlp model
   entities = []

  #creating a doc object
   doc = nlp(text)
   for ent in doc.ents:
       if ent.label_ == "PERSON": #keeping only PERSON entities and appending it to our entities list
           entities.append(
           ent.text.strip()
           )
           #.strip() gets rid of leading or trailing whitespace

   return entities

In [ ]:
#apply NER; make sure to specify the nlp model
aa_files_data["entities"] = aa_files_data["text"].apply(lambda x: run_ner(x, nlp))


In [ ]:
#explode df
aa_data_exploded = aa_files_data.explode("entities")

In [ ]:
#group by ner column; the starting and ending parentheses are just a way for Python to treat this as one
#continous line of code despite the line breaks
aa_data_deduped = (
   aa_data_exploded.groupby("entities", as_index=False)
   .agg(source_files=("file_name", lambda x: "; ".join(sorted(x.unique()))))
)

In [ ]:
#print off the dataframe to verify if it needs cleaning (spoiler: it does)
aa_data_deduped

In [ ]:
#data cleaning

#keep only rows with multiple token names (if there is no space = one word)
aa_data_deduped = aa_data_deduped[aa_data_deduped["entities"].str.contains(" ")]


#use regex and a mask “~” to get rid of rows containing any digits
aa_data_deduped = aa_data_deduped[~aa_data_deduped["entities"].str.contains(r"\d")]

# use a mask “~” to get rid of rows containing "AA"
aa_data_deduped = aa_data_deduped[~aa_data_deduped["entities"].str.contains("AA")]
